# Обзор датасета резервной одометрии

Этот notebook строит **быструю инвентаризацию всех ROS 2 bag по `metadata.yaml`**. Он не декодирует 8,8 млн сообщений и поэтому подходит для первого знакомства, выбора прогонов и подготовки train/validation/test.

Что здесь можно увидеть:

- длительность, объём и количество сообщений каждого прогона;
- наличие эталонных GNSS-потоков;
- различия между трамваями `30618` и `30639`;
- кандидатов в повторяющиеся записи, которые нельзя случайно разнести между train и test.

> GNSS разрешён только как ground truth для офлайн-аналитики, метрик и начальной выставки. В основном контуре резервной одометрии используются только ручка контроллера и скорости тележек.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from bag_analysis import (
    DATA_ROOT,
    candidate_duplicate_groups,
    scan_dataset,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

print(f"Каталог данных: {DATA_ROOT}")

## 1. Инвентаризация

In [ ]:
inventory = scan_dataset()

summary = pd.Series({
    "bag_count": len(inventory),
    "vehicles": inventory["vehicle_id"].nunique(),
    "total_duration_hours": inventory["duration_s"].sum() / 3600,
    "total_messages": inventory["message_count"].sum(),
    "total_size_mib": inventory["size_mib"].sum(),
    "median_duration_s": inventory["duration_s"].median(),
    "bags_with_any_gnss": inventory["has_any_gnss"].sum(),
    "bags_with_both_gnss": inventory["has_both_gnss"].sum(),
    "bags_without_gnss": (~inventory["has_any_gnss"]).sum(),
})

display(summary.to_frame("value"))
display(inventory.head())

### Проверки целостности

Эти проверки не доказывают корректность самих измерений, но быстро показывают неполную распаковку или неожиданное изменение структуры датасета.

In [ ]:
integrity_checks = pd.Series({
    "metadata_read_for_every_bag": len(inventory) == len(list(DATA_ROOT.glob("*/metadata.yaml"))),
    "all_have_control": inventory["control_count"].gt(0).all(),
    "all_have_front_wheel": inventory["front_wheel_count"].gt(0).all(),
    "all_have_rear_wheel": inventory["rear_wheel_count"].gt(0).all(),
    "message_counts_positive": inventory["message_count"].gt(0).all(),
    "durations_positive": inventory["duration_s"].gt(0).all(),
})

integrity_checks.to_frame("passed")

## 2. Распределения прогонов

In [ ]:
plot_data = inventory.assign(
    duration_min=inventory["duration_s"] / 60,
    gnss=inventory["has_any_gnss"].map({True: "GNSS есть", False: "GNSS отсутствует"}),
)

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

sns.histplot(plot_data, x="duration_min", bins=30, hue="vehicle_id", element="step", ax=axes[0, 0])
axes[0, 0].set(title="Длительность прогонов", xlabel="Минуты", ylabel="Количество bag")

sns.scatterplot(
    plot_data,
    x="duration_min",
    y="message_count",
    hue="gnss",
    style="vehicle_id",
    ax=axes[0, 1],
)
axes[0, 1].set(title="Длительность и число сообщений", xlabel="Минуты", ylabel="Сообщения")

sns.boxplot(plot_data, x="vehicle_id", y="duration_min", hue="gnss", ax=axes[1, 0])
axes[1, 0].set(title="Длительность по трамваям", xlabel="Трамвай", ylabel="Минуты")

availability = (
    plot_data.groupby(["vehicle_id", "gnss"], observed=True)
    .size()
    .rename("bags")
    .reset_index()
)
sns.barplot(availability, x="vehicle_id", y="bags", hue="gnss", ax=axes[1, 1])
axes[1, 1].set(title="Доступность GNSS", xlabel="Трамвай", ylabel="Количество bag")

fig.tight_layout()
plt.show()

## 3. Покрытие каналов

In [ ]:
coverage_by_vehicle = inventory.groupby("vehicle_id").agg(
    bags=("bag_id", "size"),
    duration_hours=("duration_s", lambda values: values.sum() / 3600),
    messages=("message_count", "sum"),
    control=("control_count", "sum"),
    front_wheel=("front_wheel_count", "sum"),
    rear_wheel=("rear_wheel_count", "sum"),
    master_fix=("master_fix_count", "sum"),
    master_velocity=("master_velocity_count", "sum"),
    rover_fix=("rover_fix_count", "sum"),
    rover_velocity=("rover_velocity_count", "sum"),
    bags_with_both_gnss=("has_both_gnss", "sum"),
    bags_without_gnss=("has_any_gnss", lambda values: (~values).sum()),
)

coverage_by_vehicle

## 4. Кандидаты в повторяющиеся записи

Совпадающие длительность и счётчики топиков — только **признак возможного дубликата**, а не доказательство. Перед разбиением выборки такие группы нужно подтвердить хешами `.db3` или сравнением временных диапазонов. Все bag одной подтверждённой группы должны попасть в одну часть выборки.

In [ ]:
duplicate_candidates = candidate_duplicate_groups(inventory)

print(f"Кандидатов: {len(duplicate_candidates)} bag")
print(f"Групп кандидатов: {duplicate_candidates['candidate_group'].nunique() if not duplicate_candidates.empty else 0}")

duplicate_candidates[[
    "candidate_group",
    "bag_id",
    "vehicle_id",
    "duration_s",
    "message_count",
    "has_any_gnss",
]].head(50)

## 5. Выбор прогонов для экспериментов

In [ ]:
inventory_with_class = inventory.assign(
    recommended_use=np.select(
        [
            inventory["duration_s"].lt(120) & inventory["has_any_gnss"],
            inventory["duration_s"].ge(120) & inventory["has_both_gnss"],
            ~inventory["has_any_gnss"],
        ],
        [
            "быстрая отладка с truth",
            "оценка точности и калибровка",
            "устойчивость без доступных метрик truth",
        ],
        default="ручная проверка",
    )
)

selection = (
    inventory_with_class.sort_values(["recommended_use", "duration_s"])
    [[
        "bag_id",
        "vehicle_id",
        "duration_s",
        "message_count",
        "has_master_gnss",
        "has_rover_gnss",
        "recommended_use",
    ]]
)

selection.groupby("recommended_use", group_keys=False).head(10)

## Как продолжить

1. Выберите `bag_id` из таблиц выше.
2. Откройте `explore_bag.ipynb` и замените параметр `BAG_ID`.
3. Для train/validation/test делите данные **целыми bag**, учитывайте трамвай и подтверждённые дубликаты.
4. Bag без GNSS полезны для проверки доступности и устойчивости, но не дают измерить фактическую ошибку положения или скорости.
5. Не интерпретируйте `recorded_ns - header_ns` как чистую сетевую задержку без отдельного анализа часов.